## Example 1: Basic Hash Algorithms and Generation

In [ ]:
import xxhash

# Example 1: Basic xxhash algorithms and hash generation
print("Basic xxhash algorithms and hash generation:")

# Test data
test_data = b"Hello, xxhash world!"
test_string = "Hello, xxhash world!"

print(f"Input data: '{test_string}'")
print(f"Input bytes: {len(test_data)} bytes")

# Test different xxhash algorithms
algorithms = [
    ('xxh32', xxhash.xxh32),
    ('xxh64', xxhash.xxh64),
    ('xxh3_64', xxhash.xxh3_64),
    ('xxh3_128', xxhash.xxh3_128)
]

print(f"\nHash results for different algorithms:")
for name, hasher in algorithms:
    hash_obj = hasher(test_data)
    hex_digest = hash_obj.hexdigest()
    int_digest = hash_obj.intdigest()
    
    print(f"  {name:8}: {hex_digest} (int: {int_digest})")

# Show package information
print(f"\nPackage information:")
print(f"xxhash version: {xxhash.version}")
print(f"XXHASH library version: {xxhash.XXHASH_VERSION}")
print(f"Available algorithms: {xxhash.algorithms_available}")

print("Basic hash generation completed")

## Example 2: Incremental Hashing and Streaming

In [ ]:
import xxhash
import time

# Example 2: Incremental hashing and streaming data processing
print("Incremental hashing and streaming data processing:")

# Create incremental hash objects
hash_32 = xxhash.xxh32()
hash_64 = xxhash.xxh64()
hash_3_64 = xxhash.xxh3_64()
hash_3_128 = xxhash.xxh3_128()

# Simulate streaming data processing
data_chunks = [
    b"This is the first chunk of data. ",
    b"Here comes the second chunk. ",
    b"And finally the third chunk to complete the stream."
]

print(f"Processing {len(data_chunks)} data chunks incrementally:")

# Process each chunk incrementally
for i, chunk in enumerate(data_chunks):
    print(f"  Processing chunk {i+1}: '{chunk.decode()[:30]}...'")
    
    # Update each hash with the current chunk
    hash_32.update(chunk)
    hash_64.update(chunk)
    hash_3_64.update(chunk)
    hash_3_128.update(chunk)

print(f"\nIncremental hash results:")
print(f"  xxh32:    {hash_32.hexdigest()}")
print(f"  xxh64:    {hash_64.hexdigest()}")
print(f"  xxh3_64:  {hash_3_64.hexdigest()}")
print(f"  xxh3_128: {hash_3_128.hexdigest()}")

# Compare with one-shot hashing
combined_data = b"".join(data_chunks)
print(f"\nOne-shot hash verification:")
print(f"  xxh32:    {xxhash.xxh32(combined_data).hexdigest()}")
print(f"  xxh64:    {xxhash.xxh64(combined_data).hexdigest()}")
print(f"  xxh3_64:  {xxhash.xxh3_64(combined_data).hexdigest()}")
print(f"  xxh3_128: {xxhash.xxh3_128(combined_data).hexdigest()}")

# Test with custom seeds
print(f"\nTesting with custom seeds:")
seed = 12345
seeded_hash = xxhash.xxh64(combined_data, seed=seed)
print(f"  xxh64 with seed {seed}: {seeded_hash.hexdigest()}")

print("Incremental hashing and streaming completed")

## Example 3: Hash Comparison and Validation

In [ ]:
import xxhash
import hashlib
import time

# Example 3: Hash comparison, validation, and collision detection
print("Hash comparison, validation, and collision detection:")

# Test data for comparison
test_strings = [
    "Hello World",
    "Hello world",  # Different case
    "Hello World!",  # Different punctuation
    "Hello World",  # Duplicate
    "",  # Empty string
    "A" * 1000,  # Long string
]

print(f"Testing hash consistency and comparison:")

# Generate hashes for all test strings
results = []
for i, text in enumerate(test_strings):
    data = text.encode('utf-8')
    
    hash_result = {
        'index': i,
        'text': text[:50] + "..." if len(text) > 50 else text,
        'xxh32': xxhash.xxh32(data).hexdigest(),
        'xxh64': xxhash.xxh64(data).hexdigest(),
        'xxh3_64': xxhash.xxh3_64(data).hexdigest(),
        'md5': hashlib.md5(data).hexdigest(),
    }
    results.append(hash_result)
    
    print(f"  {i+1}. '{hash_result['text']}'")
    print(f"     xxh32:   {hash_result['xxh32']}")
    print(f"     xxh64:   {hash_result['xxh64']}")
    print(f"     xxh3_64: {hash_result['xxh3_64']}")
    print()

# Test hash consistency (same input should produce same hash)
print("Testing hash consistency:")
test_data = b"Consistency test data"
hashes_1 = [xxhash.xxh64(test_data).hexdigest() for _ in range(5)]
all_same = all(h == hashes_1[0] for h in hashes_1)
print(f"  5 hash computations of same data: {'✓ All identical' if all_same else '✗ Inconsistent'}")

# Test hash uniqueness (different inputs should produce different hashes)
print(f"\nTesting hash uniqueness:")
unique_xxh64 = set(r['xxh64'] for r in results)
unique_xxh32 = set(r['xxh32'] for r in results)
total_inputs = len([r for r in results if r['text'] != 'Hello World' or results.index(r) == 0])  # Exclude duplicate

print(f"  xxh64 unique hashes: {len(unique_xxh64)} out of {len(results)} inputs")
print(f"  xxh32 unique hashes: {len(unique_xxh32)} out of {len(results)} inputs")
print(f"  Expected unique: {total_inputs} (excluding intentional duplicate)")

# Test hash validation function
def validate_hash_integrity(data, expected_hash, algorithm='xxh64'):
    """Validate data integrity using hash comparison"""
    if algorithm == 'xxh64':
        computed_hash = xxhash.xxh64(data).hexdigest()
    elif algorithm == 'xxh32':
        computed_hash = xxhash.xxh32(data).hexdigest()
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")
    
    return computed_hash == expected_hash, computed_hash

print(f"\nTesting hash validation:")
original_data = b"Important data to validate"
original_hash = xxhash.xxh64(original_data).hexdigest()

# Test with unchanged data
valid, computed = validate_hash_integrity(original_data, original_hash)
print(f"  Original data validation: {'✓ Valid' if valid else '✗ Invalid'}")

# Test with corrupted data
corrupted_data = b"Important data to validatE"  # Changed last character
valid, computed = validate_hash_integrity(corrupted_data, original_hash)
print(f"  Corrupted data validation: {'✓ Valid' if valid else '✗ Invalid (expected)'}")
print(f"    Original:  {original_hash}")
print(f"    Computed:  {computed}")

print("Hash comparison and validation completed")

## Example 4: Performance Benchmarking

In [ ]:
import xxhash
import hashlib
import time
import random
import string

# Example 4: Performance benchmarking and algorithm comparison
print("Performance benchmarking and algorithm comparison:")

def generate_test_data(size_kb):
    """Generate random test data of specified size"""
    size_bytes = size_kb * 1024
    return ''.join(random.choices(string.ascii_letters + string.digits, k=size_bytes)).encode()

def benchmark_hash_algorithm(hash_func, data, iterations=100):
    """Benchmark a hash algorithm with multiple iterations"""
    start_time = time.time()
    
    for _ in range(iterations):
        if hasattr(hash_func, '__call__'):
            # For xxhash functions that return hash objects
            hash_func(data)
        else:
            # For hashlib functions
            hash_func(data).hexdigest()
    
    end_time = time.time()
    total_time = end_time - start_time
    
    return {
        'total_time': total_time,
        'avg_time': total_time / iterations,
        'hashes_per_sec': iterations / total_time if total_time > 0 else float('inf'),
        'mb_per_sec': (len(data) * iterations / (1024 * 1024)) / total_time if total_time > 0 else float('inf')
    }

# Test with different data sizes
data_sizes = [1, 10, 100, 1000]  # KB
algorithms = {
    'xxh32': xxhash.xxh32,
    'xxh64': xxhash.xxh64,
    'xxh3_64': xxhash.xxh3_64,
    'xxh3_128': xxhash.xxh3_128,
    'md5': hashlib.md5,
    'sha1': hashlib.sha1,
    'sha256': hashlib.sha256
}

print("Benchmarking hash algorithms with different data sizes:")
print("=" * 70)

results = {}
for size_kb in data_sizes:
    print(f"\nData size: {size_kb} KB")
    test_data = generate_test_data(size_kb)
    
    results[size_kb] = {}
    for name, hash_func in algorithms.items():
        try:
            # Run fewer iterations for larger data to keep benchmark reasonable
            iterations = max(10, 100 // size_kb) if size_kb >= 100 else 100
            
            benchmark = benchmark_hash_algorithm(hash_func, test_data, iterations)
            results[size_kb][name] = benchmark
            
            print(f"  {name:8}: {benchmark['avg_time']*1000:8.3f} ms/hash, "
                  f"{benchmark['hashes_per_sec']:8.1f} hashes/sec, "
                  f"{benchmark['mb_per_sec']:8.1f} MB/sec")
                  
        except Exception as e:
            print(f"  {name:8}: Error - {e}")
            results[size_kb][name] = None

# Performance comparison analysis
print(f"\nPerformance Analysis:")
print("=" * 50)

for size_kb in data_sizes:
    if size_kb not in results:
        continue
        
    print(f"\n{size_kb} KB data analysis:")
    
    # Find fastest algorithm by throughput (MB/sec)
    valid_results = {k: v for k, v in results[size_kb].items() if v is not None}
    if valid_results:
        fastest = max(valid_results.items(), key=lambda x: x[1]['mb_per_sec'])
        slowest = min(valid_results.items(), key=lambda x: x[1]['mb_per_sec'])
        
        print(f"  Fastest: {fastest[0]} ({fastest[1]['mb_per_sec']:.1f} MB/sec)")
        print(f"  Slowest: {slowest[0]} ({slowest[1]['mb_per_sec']:.1f} MB/sec)")
        
        if slowest[1]['mb_per_sec'] > 0:
            speedup = fastest[1]['mb_per_sec'] / slowest[1]['mb_per_sec']
            print(f"  Speedup: {speedup:.1f}x faster")

# Memory efficiency test
print(f"\nMemory efficiency test:")
small_data = b"test data for memory efficiency"

# Test hash object sizes
hash_objects = {
    'xxh32': xxhash.xxh32(small_data),
    'xxh64': xxhash.xxh64(small_data),
    'xxh3_64': xxhash.xxh3_64(small_data),
    'xxh3_128': xxhash.xxh3_128(small_data)
}

for name, hash_obj in hash_objects.items():
    hex_digest = hash_obj.hexdigest()
    int_digest = hash_obj.intdigest() if hasattr(hash_obj, 'intdigest') else 'N/A'
    
    print(f"  {name:8}: hex={len(hex_digest)} chars, int={len(str(int_digest)) if int_digest != 'N/A' else 'N/A'} digits")

print(f"\nKey findings:")
print("- xxhash algorithms typically outperform cryptographic hashes for speed")
print("- xxh3_64 and xxh3_128 often show best performance on modern CPUs") 
print("- Cryptographic hashes (SHA256) provide security but are slower")
print("- xxhash is ideal for hash tables, checksums, and non-cryptographic use cases")

print("Performance benchmarking completed")

## Example 5: Production File Processing Workflows

In [ ]:
import xxhash
import json
import os
import tempfile
import time
from collections import defaultdict

# Example 5: Production patterns and file processing workflows
print("Production xxhash patterns and file processing workflows:")

class HashProcessor:
    """Production-ready hash processor with file operations and caching"""
    
    def __init__(self):
        self.stats = {
            'files_processed': 0,
            'bytes_processed': 0,
            'errors': 0,
            'cache_hits': 0,
            'processing_time': 0
        }
        self.hash_cache = {}
        self.supported_algorithms = ['xxh32', 'xxh64', 'xxh3_64', 'xxh3_128']
    
    def hash_data(self, data, algorithm='xxh64', seed=None):
        """Hash data using specified xxhash algorithm"""
        if algorithm not in self.supported_algorithms:
            raise ValueError(f"Unsupported algorithm: {algorithm}")
        
        hash_kwargs = {'seed': seed} if seed is not None else {}
        
        if algorithm == 'xxh32':
            return xxhash.xxh32(data, **hash_kwargs).hexdigest()
        elif algorithm == 'xxh64':
            return xxhash.xxh64(data, **hash_kwargs).hexdigest()
        elif algorithm == 'xxh3_64':
            # xxh3 doesn't use seed parameter the same way
            return xxhash.xxh3_64(data).hexdigest()
        elif algorithm == 'xxh3_128':
            return xxhash.xxh3_128(data).hexdigest()
    
    def hash_file_content(self, content, filename, algorithm='xxh64'):
        """Hash file content with caching and statistics"""
        start_time = time.time()
        
        # Create cache key
        cache_key = f"{filename}:{algorithm}:{len(content)}"
        
        # Check cache first
        if cache_key in self.hash_cache:
            self.stats['cache_hits'] += 1
            return self.hash_cache[cache_key], True  # True indicates cache hit
        
        try:
            # Compute hash
            hash_result = self.hash_data(content, algorithm)
            
            # Update cache and statistics
            self.hash_cache[cache_key] = hash_result
            self.stats['files_processed'] += 1
            self.stats['bytes_processed'] += len(content)
            self.stats['processing_time'] += time.time() - start_time
            
            return hash_result, False  # False indicates fresh computation
            
        except Exception as e:
            self.stats['errors'] += 1
            raise e
    
    def create_integrity_manifest(self, file_contents, algorithm='xxh64'):
        """Create integrity manifest for multiple files"""
        manifest = {
            'algorithm': algorithm,
            'created_at': time.time(),
            'files': {}
        }
        
        for filename, content in file_contents.items():
            try:
                file_hash, from_cache = self.hash_file_content(content, filename, algorithm)
                manifest['files'][filename] = {
                    'hash': file_hash,
                    'size': len(content),
                    'from_cache': from_cache
                }
            except Exception as e:
                manifest['files'][filename] = {
                    'error': str(e),
                    'size': len(content)
                }
        
        return manifest
    
    def verify_integrity(self, file_contents, manifest):
        """Verify file integrity against manifest"""
        verification_results = {
            'verified': 0,
            'failed': 0,
            'missing': 0,
            'errors': 0,
            'details': {}
        }
        
        algorithm = manifest.get('algorithm', 'xxh64')
        
        for filename, expected_data in manifest['files'].items():
            if 'error' in expected_data:
                verification_results['errors'] += 1
                verification_results['details'][filename] = 'Original hash failed'
                continue
            
            if filename not in file_contents:
                verification_results['missing'] += 1
                verification_results['details'][filename] = 'File missing'
                continue
            
            try:
                current_hash, _ = self.hash_file_content(
                    file_contents[filename], filename, algorithm
                )
                
                if current_hash == expected_data['hash']:
                    verification_results['verified'] += 1
                    verification_results['details'][filename] = 'OK'
                else:
                    verification_results['failed'] += 1
                    verification_results['details'][filename] = f'Hash mismatch: expected {expected_data["hash"][:16]}..., got {current_hash[:16]}...'
                    
            except Exception as e:
                verification_results['errors'] += 1
                verification_results['details'][filename] = f'Verification error: {e}'
        
        return verification_results
    
    def get_processing_statistics(self):
        """Get detailed processing statistics"""
        stats = self.stats.copy()
        if stats['files_processed'] > 0:
            stats['avg_processing_time'] = stats['processing_time'] / stats['files_processed']
            stats['avg_file_size'] = stats['bytes_processed'] / stats['files_processed']
            stats['cache_hit_rate'] = stats['cache_hits'] / (stats['files_processed'] + stats['cache_hits']) * 100
        else:
            stats['avg_processing_time'] = 0
            stats['avg_file_size'] = 0
            stats['cache_hit_rate'] = 0
        
        return stats

# Create processor and demonstrate production workflows
processor = HashProcessor()

print("Creating sample file set for processing:")

# Create sample files for demonstration
sample_files = {
    'config.json': b'{"app_name": "HashDemo", "version": "1.0", "debug": true}',
    'data.csv': b'id,name,value\n1,Alice,100\n2,Bob,200\n3,Charlie,300',
    'readme.txt': b'This is a sample application for demonstrating hash processing workflows.',
    'script.py': b'#!/usr/bin/env python3\nprint("Hello, xxhash world!")\n# Hash processing demo',
    'empty.txt': b'',
    'large_file.dat': b'X' * 10000  # 10KB of data
}

print(f"Sample file set: {len(sample_files)} files, {sum(len(content) for content in sample_files.values())} total bytes")

# Create integrity manifest
print(f"\nCreating integrity manifest:")
manifest = processor.create_integrity_manifest(sample_files, algorithm='xxh64')

print(f"Manifest created with {len(manifest['files'])} file entries")
for filename, file_data in manifest['files'].items():
    if 'hash' in file_data:
        print(f"  {filename}: {file_data['hash'][:16]}... ({file_data['size']} bytes)")
    else:
        raise AssertionError(f"hashing {filename} failed: {file_data.get('error', 'Unknown error')}")

# Reference vector: XXH64 of empty input, seed 0
assert manifest['files']['empty.txt']['hash'] == 'ef46db3751d8e999', f"xxh64(b'') = {manifest['files']['empty.txt']['hash']}"

# Test integrity verification with original files
print(f"\nVerifying integrity of original files:")
processor.hash_cache.clear()  # re-hash the bytes: a verification served from the cache it checks proves nothing
verification_1 = processor.verify_integrity(sample_files, manifest)
print(f"Verification results: {verification_1['verified']} verified, {verification_1['failed']} failed, {verification_1['missing']} missing")
assert (verification_1['verified'], verification_1['failed'], verification_1['missing'], verification_1['errors']) == (len(sample_files), 0, 0, 0), f"unmodified files must all verify: {verification_1}"

# Test with modified files
print(f"\nTesting with modified files:")
modified_files = sample_files.copy()
modified_files['config.json'] = b'{"app_name": "HashDemo", "version": "2.0", "debug": false}'  # Modified
del modified_files['empty.txt']  # Removed file

verification_2 = processor.verify_integrity(modified_files, manifest)
print(f"Verification results: {verification_2['verified']} verified, {verification_2['failed']} failed, {verification_2['missing']} missing")
assert (verification_2['verified'], verification_2['failed'], verification_2['missing'], verification_2['errors']) == (len(sample_files) - 2, 1, 1, 0), f"one modified + one removed file must be caught: {verification_2}"

# Show detailed verification results
print(f"\nDetailed verification results:")
for filename, result in verification_2['details'].items():
    status = "✓" if result == "OK" else "✗"
    print(f"  {status} {filename}: {result}")

# Performance comparison across algorithms
print(f"\nTesting different hash algorithms:")
algorithms = ['xxh32', 'xxh64', 'xxh3_64', 'xxh3_128']
large_data = b'Performance test data ' * 1000  # ~23KB

for algorithm in algorithms:
    start_time = time.time()
    hash_result = processor.hash_data(large_data, algorithm)
    end_time = time.time()
    
    processing_time = (end_time - start_time) * 1000  # Convert to milliseconds
    throughput = len(large_data) / (1024 * 1024) / (end_time - start_time)  # MB/s
    
    print(f"  {algorithm:8}: {hash_result[:16]}... ({processing_time:.2f} ms, {throughput:.1f} MB/s)")

# Show final processing statistics
print(f"\nFinal processing statistics:")
final_stats = processor.get_processing_statistics()
for key, value in final_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

print(f"\nProduction features demonstrated:")
print("- File content integrity verification with manifests")
print("- Hash result caching for performance optimization")
print("- Multiple xxhash algorithm support (xxh32, xxh64, xxh3_64, xxh3_128)")
print("- Comprehensive error handling and statistics tracking")
print("- Batch file processing with detailed verification reporting")
print("- Performance monitoring and throughput analysis")